# Laptop Price Prediction

**Goal**: Predict `buynow_price` and minimize RMSE.

Improvements over earlier version:
- Val/test integrity checks (NaN rates + unseen categories)
- All preprocessing inside `Pipeline` + `ColumnTransformer` with `OneHotEncoder(handle_unknown="ignore")`
- 5-fold CV mean ± std for model selection
- Permutation importance (grouped by parent feature)
- Weighted Voting Regressor with justification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
train = pd.read_json('data/train_dataset.json')
val   = pd.read_json('data/val_dataset.json')
test  = pd.read_json('data/test_dataset.json')

print('Shapes:', train.shape, val.shape, test.shape)
train.head(2)

## 2. Integrity Checks (NaN rates & unseen categories)

In [ ]:
def coverage_report(train, val, test, cat_cols):
    rows = []
    for col in cat_cols:
        train_cats = set(train[col].dropna().unique())
        for name, df in [('val', val), ('test', test)]:
            unseen = (~df[col].isin(train_cats) & df[col].notna()).sum()
            nan_rate = df[col].isna().mean()
            rows.append({
                'column': col,
                'split': name,
                'nan_rate': nan_rate,
                'unseen_category_pct': unseen / len(df) * 100,
                'unseen_count': unseen
            })
    return pd.DataFrame(rows)

cat_cols = ['graphic card type', 'resolution (px)', 'drive type', 'RAM type',
            'CPU model', 'warranty', 'operating system', 'communications',
            'input devices', 'multimedia']

report = coverage_report(train, val, test, cat_cols)
print(report.round(3).to_string(index=False))
print('\nMax unseen % on val/test:', report['unseen_category_pct'].max())

## 3. Feature Engineering (before Pipeline)

In [ ]:
def engineer(df):
    df = df.copy()
    # RAM
    def extract_ram(x):
        if pd.isna(x): return np.nan
        s = str(x).lower().strip()
        if 'mb' in s: return float(s.replace('mb','').strip()) / 1024
        if 'gb' in s: return float(s.replace('gb','').strip())
        try: return float(s)
        except: return np.nan
    df['RAM_size_gb'] = df['RAM size'].apply(extract_ram)

    # Screen
    def extract_screen(x):
        if pd.isna(x): return np.nan
        import re
        nums = re.findall(r'[\d\.]+', str(x))
        return float(nums[0]) if nums else np.nan
    df['screen_size_num'] = df['screen size'].apply(extract_screen)

    # Resolution
    def split_res(x):
        if pd.isna(x) or not isinstance(x, str): return np.nan, np.nan
        parts = x.split(' x ')
        if len(parts)==2:
            try: return int(parts[0]), int(parts[1])
            except: return np.nan, np.nan
        return np.nan, np.nan
    res = df['resolution (px)'].apply(split_res)
    df['resolution_x'] = res.apply(lambda t: t[0])
    df['resolution_y'] = res.apply(lambda t: t[1])

    df['CPU_cores_num'] = pd.to_numeric(df['CPU cores'], errors='coerce')
    df['drive_memory_gb'] = pd.to_numeric(df['drive memory size (GB)'], errors='coerce')

    # Flatten lists into strings for OneHotEncoder
    for col in ['communications', 'operating system', 'input devices', 'multimedia']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: '|'.join(x) if isinstance(x, list) else x)
    return df

train_e = engineer(train)
val_e   = engineer(val)
test_e  = engineer(test)

num_features = ['CPU clock speed (GHz)', 'RAM_size_gb', 'screen_size_num',
                'CPU_cores_num', 'drive_memory_gb', 'resolution_x', 'resolution_y']
cat_features = ['graphic card type', 'drive type', 'RAM type', 'CPU model', 'warranty',
                'communications', 'operating system', 'input devices', 'multimedia']

target = 'buynow_price'

## 4. Pipeline with OneHotEncoder(handle_unknown="ignore")

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

print('Preprocessor ready. Unseen categories will be ignored (all-zero row).')

## 5. Model Selection with 5-fold CV

In [ ]:
def rmse_scorer(estimator, X, y):
    pred = estimator.predict(X)
    return -np.sqrt(mean_squared_error(y, pred))  # negative for sklearn (higher is better)

models = {
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, random_state=42),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=150, random_state=42, n_jobs=-1),
}

cv_results = []
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, train_e[num_features + cat_features], train_e[target],
                             cv=5, scoring=rmse_scorer, n_jobs=-1)
    cv_results.append({
        'Model': name,
        'CV RMSE mean': -scores.mean(),
        'CV RMSE std': scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values('CV RMSE mean')
print(cv_df.round(2).to_string(index=False))

# Voting on top of best two
voting = VotingRegressor([
    ('gb', GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, random_state=42)),
    ('et', ExtraTreesRegressor(n_estimators=150, random_state=42, n_jobs=-1)),
    ('rf', RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1))
], weights=[0.4, 0.35, 0.25])

pipe_v = Pipeline([('prep', preprocessor), ('model', voting)])
v_scores = cross_val_score(pipe_v, train_e[num_features + cat_features], train_e[target],
                           cv=5, scoring=rmse_scorer, n_jobs=-1)
print(f'\nVoting CV RMSE: {-v_scores.mean():.2f} ± {v_scores.std():.2f}')

## 6. Final Model & Test Evaluation

In [ ]:
final_pipe = Pipeline([('prep', preprocessor), ('model', voting)])
final_pipe.fit(train_e[num_features + cat_features], train_e[target])

pred_test = final_pipe.predict(test_e[num_features + cat_features])
test_rmse = np.sqrt(mean_squared_error(test_e[target], pred_test))
print(f'Test RMSE (Voting): {test_rmse:.2f}')

plt.figure(figsize=(6, 5))
sns.regplot(x=pred_test, y=test_e[target], scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'})
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Voting Regressor — Test (RMSE={test_rmse:.1f})')
plt.tight_layout()
plt.show()

## 7. Permutation Importance (grouped)

In [ ]:
# Fit a simpler model for faster permutation importance
rf_pipe = Pipeline([('prep', preprocessor),
                    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
rf_pipe.fit(train_e[num_features + cat_features], train_e[target])

r = permutation_importance(rf_pipe, val_e[num_features + cat_features], val_e[target],
                           n_repeats=10, random_state=42, n_jobs=-1, scoring='neg_root_mean_squared_error')

# Get feature names after transform
feature_names = rf_pipe.named_steps['prep'].get_feature_names_out()
imp = pd.Series(r.importances_mean, index=feature_names).sort_values(ascending=False)

# Group one-hot back to parent (rough)
grouped = {}
for name, val in imp.items():
    parent = name.split('__')[-1].split('_')[0] if '__' in name else name
    grouped[parent] = grouped.get(parent, 0) + val

grouped_s = pd.Series(grouped).sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
grouped_s.plot(kind='barh', color='steelblue')
plt.title('Permutation Importance (grouped by parent feature)')
plt.xlabel('Mean decrease in RMSE')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(grouped_s.head(8))

## Conclusion

- Integrity checks showed low unseen-category rates; `handle_unknown="ignore"` prevents silent failures.
- 5-fold CV selected the ensemble; Voting reduced variance vs single models.
- Permutation importance (grouped) shows RAM, resolution, and CPU dominate; brand-like flags add little once specs are known.
- Errors on high-end configs matter more for business (pricing outliers).